# Block-Wise Shrinkage GMRF: Parameter Estimation Demo

This notebook implements the **Parameter Estimation phase** of a custom hierarchical reconciliation algorithm (Block-Wise Shrinkage GMRF).

## Objectives
1.  **Generate Data**: Create a larger strict hierarchy (Country -> Region -> State) with 25 series.
2.  **Base Forecasts**: Use AutoARIMA to obtain residuals.
3.  **Phase 1: Hierarchy Decomposition**: Identify Parent-Children families.
4.  **Phase 2: Local Robust Estimation**: Apply Ledoit-Wolf shrinkage to each family block.
5.  **Phase 3: GMRF Parameter Extraction**: Sparsify the graph by extracting $\phi$ and $\sigma^2$ parameters.
6.  **Verification**: Visualize $\phi$ vs Sample Correlations.

In [66]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.covariance import LedoitWolf
from numpy.linalg import inv

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from hierarchicalforecast.utils import aggregate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Synthetic Data Generation (Expanded Hierarchy)

Structure: **Country (1) -> Region (4) -> State (5 per Region)** = 25 Total Series.

In [67]:
# Configuration
n_periods = 300
freq = 'D'
start_date = '2023-01-01'
seed = 42

np.random.seed(seed)
dates = pd.date_range(start=start_date, periods=n_periods, freq=freq)

# --- Define Expanded Strict Hierarchy ---
regions = ['RegionA', 'RegionB', 'RegionC', 'RegionD']
states_per_region = 5

data_rows = []

# Helper to generate random series
def generate_series(n, start_val=100, drift=0.1, noise_level=5):
    return np.cumsum(np.random.normal(drift, noise_level, n)) + start_val

# Generate Bottom Level (State) Data
for region in regions:
    for i in range(1, states_per_region + 1):
        state_name = f"{region}_State{i}"
        values = generate_series(n_periods)
        
        for d, val in zip(dates, values):
            data_rows.append({
                'Country': 'Total',
                'Region': region,
                'State': state_name,
                'ds': d,
                'y': val
            })

flat_df = pd.DataFrame(data_rows)

# Aggregate
spec = [
    ['Country'],
    ['Country', 'Region'],
    ['Country', 'Region', 'State']
]

Y_df, S_df, tags = aggregate(flat_df, spec)
# Note: S_df usually comes with 'unique_id' column from aggregate, avoiding reset_index issues seen before.

print(f"Total series: {len(Y_df['unique_id'].unique())}")
display(S_df.head())

Total series: 25


,unique_id,Total/RegionA/RegionA_State1,Total/RegionA/RegionA_State2,Total/RegionA/RegionA_State3,Total/RegionA/RegionA_State4,Total/RegionA/RegionA_State5,Total/RegionB/RegionB_State1,Total/RegionB/RegionB_State2,Total/RegionB/RegionB_State3,Total/RegionB/RegionB_State4,...,Total/RegionC/RegionC_State1,Total/RegionC/RegionC_State2,Total/RegionC/RegionC_State3,Total/RegionC/RegionC_State4,Total/RegionC/RegionC_State5,Total/RegionD/RegionD_State1,Total/RegionD/RegionD_State2,Total/RegionD/RegionD_State3,Total/RegionD/RegionD_State4,Total/RegionD/RegionD_State5
0,Total,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,Total/RegionA,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Total/RegionB,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Total/RegionC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Total/RegionD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0


## 2. Base Forecasts & Residuals Calculation

In [68]:
# Forecast
Y_train_df = Y_df  # Use full data for this param estimation demo (or split if strictly needed)
# Ensure strictly the required columns to avoid errors
Y_train_df = Y_train_df[['unique_id', 'ds', 'y']]

fcst = StatsForecast(
    models=[AutoARIMA(season_length=7)],
    freq=freq, n_jobs=-1
)

# Fit and get in-sample residuals
# To get residuals for the training set, we use fit_predict or cross_validation, 
# or simply fit and then subtract fitted from actuals.
# Here we'll use forecast_fitted_values after a mock forecast call or just accessing the object properties if available.
# StatsForecast pattern for residuals: fit -> predict_in_sample or similar.
# For simplicity: forecast(fitted=True) on the whole dataset (h=0 effectively for fit, but we need structure)
# We will forecast 1 step just to trigger the fitted values computation easily

# We will forecast 1 step just to trigger the fitted values computation easily

fcst.forecast(df=Y_train_df, h=1, fitted=True)
fitted_vals = fcst.forecast_fitted_values()

# Calculate Residuals
residuals_long = fitted_vals.merge(Y_train_df, on=['unique_id', 'ds'], suffixes=('_hat', ''))
residuals_long['resid'] = residuals_long['y'] - residuals_long['AutoARIMA']

# Pivot to Wide Matrix (T x N)
E_df = residuals_long.pivot(index='ds', columns='unique_id', values='resid').dropna()
print("Residuals Matrix Shape:", E_df.shape)
display(E_df.head())

Residuals Matrix Shape: (300, 25)


unique_id,Total,Total/RegionA,Total/RegionA/RegionA_State1,Total/RegionA/RegionA_State2,Total/RegionA/RegionA_State3,Total/RegionA/RegionA_State4,Total/RegionA/RegionA_State5,Total/RegionB,Total/RegionB/RegionB_State1,Total/RegionB/RegionB_State2,...,Total/RegionC/RegionC_State2,Total/RegionC/RegionC_State3,Total/RegionC/RegionC_State4,Total/RegionC/RegionC_State5,Total/RegionD,Total/RegionD/RegionD_State1,Total/RegionD/RegionD_State2,Total/RegionD/RegionD_State3,Total/RegionD/RegionD_State4,Total/RegionD/RegionD_State5
ds,,,,,,,,,,,,,,,,,,,,,
2023-01-01,2.003488,0.503378,0.102584,0.095955,0.103385,0.101352,0.100168,0.514105,0.103992,0.109606,...,0.101793,0.042924,0.097319,0.101741,26.838774,0.095306,0.102635,0.105244,0.094161,0.100395
2023-01-02,-37.749208,-13.431754,-0.591322,-2.700905,-5.010513,-2.444618,-2.605062,-0.726285,-2.655929,-0.203304,...,-4.589628,-0.136587,-0.550301,-4.927994,-0.192063,-6.662543,-0.186977,2.608425,-1.886616,-0.559247
2023-01-03,11.416685,10.863173,3.338443,3.836468,3.948343,-0.345698,0.153453,-4.230375,-3.990994,-3.442034,...,1.183125,10.025178,8.445348,-4.404650,-3.682699,-7.817942,-6.877243,1.341626,2.771824,0.746150
2023-01-04,21.459723,25.338961,7.715149,3.151851,6.378503,5.868792,2.258456,1.849467,0.083128,-7.468572,...,-5.028763,-2.784733,-4.612790,-4.325670,1.720696,2.164995,-3.439555,0.490589,0.555468,-2.398461
2023-01-05,12.259289,0.776717,-1.070767,-0.004508,1.667488,0.461650,-0.213734,-5.502023,-0.750923,-8.915698,...,5.604107,1.498086,8.173894,1.075725,-1.240924,-0.970338,-1.213734,2.101179,-4.157700,-1.495460


## Phase 1: Hierarchy Decomposition (Identify Families)

We identify "Families" where a family consists of a Parent node and its direct Children.

In [69]:
def get_families(S_in):
    # S_in is the summing matrix dataframe. 
    # In hierarchicalforecast, index usually contains all nodes, columns are bottom nodes.
    # However, 'aggregate' output S_df might be different. Let's inspect S_df structure from previous cell.
    # We will assume node names follow 'Parent/Child' naming convention from aggregate.
    
    # Extract all node names from unique_id column
    if 'unique_id' in S_in.columns:
        nodes = S_in['unique_id'].unique()
    else:
        nodes = S_in.index.unique()
        
    families = {}
    
    for node in nodes:
        # Root is 'Total'.
        # Separator is '/'. Children will have 'Parent/' prefix.
        # Logic: 
        # 1. Split node by '/'
        # 2. Reconstruct parent name.
        
        parts = node.split('/')
        if len(parts) == 1:
            # Top level, has no parent in this logic (or parent is None)
            continue
            
        parent = '/'.join(parts[:-1])
        
        if parent not in families:
            families[parent] = []
        families[parent].append(node)
    
    # Formatting as list of dicts
    family_list = [{'parent': k, 'children': v} for k, v in families.items()]
    return family_list

families = get_families(S_df)
print(f"Total Number of Families Found: {len(families)}")
for f in families:
    print(f"Parent: {f['parent']}, Children Count: {len(f['children'])}")

Total Number of Families Found: 5
Parent: Total, Children Count: 4
Parent: Total/RegionA, Children Count: 5
Parent: Total/RegionB, Children Count: 5
Parent: Total/RegionC, Children Count: 5
Parent: Total/RegionD, Children Count: 5


## Phase 2: Local Robust Estimation (The "Shrink" Step)

We calculate the shrunk covariance matrix for each family using Ledoit-Wolf.

In [70]:
local_estimates = {}

for i, family in enumerate(families):
    p = family['parent']
    children = family['children']
    
    # Columns for this family: Parent + Children
    cols = [p] + children
    
    # Extract subset of residuals
    # Ensure columns exist (AutoARIMA might fail for some, though unlikely with synthetic)
    valid_cols = [c for c in cols if c in E_df.columns]
    E_local = E_df[valid_cols]
    
    # Fit LedoitWolf
    lw = LedoitWolf()
    lw.fit(E_local)
    
    local_sigma = lw.covariance_
    shrinkage = lw.shrinkage_
    
    local_estimates[p] = {
        'sigma': local_sigma,
        'nodes': valid_cols,
        'shrinkage': shrinkage
    }
    
    if i == 0:
        print(f"Verification - First Family ({p}) Shrinkage Coefficient: {shrinkage:.6f}")

Verification - First Family (Total) Shrinkage Coefficient: 0.016480


## Phase 3: GMRF Parameter Extraction (The "Sparsify" Step)

We calculate the GMRF parameters $\phi$ and $\sigma^2$ using the inverse of the local covariance (Precision matrix).

In [71]:
gmrf_params = {}
sample_correlations = []
phi_values = []
labels = []

for parent, est in local_estimates.items():
    sigma = est['sigma']
    nodes = est['nodes']
    
    # Invert to get Precision Matrix
    try:
        precision = inv(sigma)
    except np.linalg.LinAlgError:
        print(f"Warning: Matrix inversion failed for {parent}")
        continue
    
    # Parent index is usually 0 since we constructed cols = [p] + children
    p_idx = 0
    prec_pp = precision[p_idx, p_idx]
    
    for i, child_node in enumerate(nodes):
        if i == p_idx: continue # Skip parent itself
        
        # Extract Parameters
        # phi_i = -1 * (Prec[p, i] / Prec[i, i])
        prec_pi = precision[p_idx, i]
        prec_ii = precision[i, i]
        
        phi = -1 * (prec_pi / prec_ii)
        
        # sigma2_i = Prec[i, i] - (phi^2 * Prec[p, p])
        # Note: prompt formula might imply conditional variance derivation. 
        sigma2 = prec_ii - (phi**2 * prec_pp)
        
        gmrf_params[child_node] = {
            'parent': parent,
            'phi': phi,
            'sigma2': sigma2
        }
        
        # --- Data for Visualization ---
        # Calculate standard sample correlation between Parent and Child
        # We use the raw residuals E_local for this
        # Correlation matrix of the current block
        corr_matrix = np.corrcoef(E_df[parent], E_df[child_node])
        sample_corr = corr_matrix[0, 1]
        
        sample_correlations.append(sample_corr)
        phi_values.append(phi)
        labels.append(child_node)

print("GMRF Parameters Extracted. Sample (first 3):")
for k, v in list(gmrf_params.items())[:3]:
    print(f"{k}: {v}")

GMRF Parameters Extracted. Sample (first 3):
Total/RegionA: {'parent': 'Total', 'phi': 0.8033192168716093, 'sigma2': 0.018249584733879734}
Total/RegionB: {'parent': 'Total', 'phi': 0.8085047136920656, 'sigma2': 0.019146605858665358}
Total/RegionC: {'parent': 'Total', 'phi': 0.8206152862520545, 'sigma2': 0.016403769899567922}


## Visualization (Research Proof)

Comparison: **Standard Correlation vs GMRF $\phi$ Parameter**.
We expect the GMRF $\phi$ values to be related to, but distinct from, raw correlations (often dampened or cleaner due to the conditional independence assumption structure).

In [76]:
import plotly.express as px
import plotly.graph_objects as go

# Create DataFrame for Plotly
plot_df = pd.DataFrame({
    'Sample Correlation': sample_correlations,
    'GMRF Phi': phi_values,
    'Series': labels,
    'Parent': [gmrf_params[l]['parent'] for l in labels]
})

fig = px.scatter(plot_df, x='Sample Correlation', y='GMRF Phi', 
                 color='Parent',
                 title='Research Proof: Standard Correlation vs GMRF Phi Parameter',
                 hover_data=['Series', 'Parent'])

# Add identity line
fig.add_shape(type='line',
    x0=-1, y0=-1, x1=1, y1=1,
    line=dict(color='Red', width=2, dash='dot'),
)

# Add axis lines
fig.add_hline(y=0, line_width=1, line_color='gray', line_dash='dash')
fig.add_vline(x=0, line_width=1, line_color='gray', line_dash='dash')

fig.update_traces(textposition='top center')
fig.update_layout(
    height=600,
    width=900,
    xaxis_title='Standard Sample Correlation',
    yaxis_title='GMRF Phi Parameter'
)

fig.show()